# Consumers & Consumer Groups

## What's covered

- The poll loop — the one shape every consumer takes
- Consumer groups — how parallelism and partition assignment actually work
- The group coordinator and the heartbeat
- The rebalance protocol — eager vs cooperative (incremental)
- Assignment strategies — `range`, `round-robin`, `sticky`, `cooperative-sticky`
- Offsets revisited — where they live (`__consumer_offsets`), how they map to consumer position
- `auto.offset.reset` — `earliest`, `latest`, `none`
- Commit modes — auto vs manual, sync vs async
- Delivery semantics on the consumer side — at-most-once, at-least-once, exactly-once
- `isolation.level=read_committed` — the consumer half of the exactly-once recipe
- Static membership (`group.instance.id`) — surviving rolling restarts without rebalances
- Timeouts — `session.timeout.ms`, `heartbeat.interval.ms`, `max.poll.interval.ms`
- `assign()` vs `subscribe()` — when you want manual control
- `seek()` — replay, skip ahead, time-based positioning
- `pause()` / `resume()` — back-pressure for the consumer side
- Common gotchas

## The poll loop

Every Kafka consumer, in every language, follows the same shape:

```text
  configure → subscribe → loop { poll → process → commit } → close
```

`poll(timeout)` does four things in one call:

1. Sends the heartbeat that tells the group coordinator "I'm still alive."
2. Participates in any rebalance the coordinator is running.
3. Fetches records from the brokers that lead the partitions you own.
4. Returns one record (or `None` on timeout) to your code.

**The single most important rule of consumer programming:** *keep calling `poll()`*. If your processing loop takes too long between polls, the coordinator decides you're dead, kicks you out of the group, and triggers a rebalance. We'll see the exact timeout below.

## Setup

Same broker as before. We'll reuse `foundations-demo` and produce some fresh records so each demo starts from a known state.

In [ ]:
from confluent_kafka import Producer, Consumer, TopicPartition, OFFSET_BEGINNING, OFFSET_END
from confluent_kafka.admin import AdminClient, NewTopic

BOOTSTRAP = "localhost:9092"
TOPIC = "foundations-demo"

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})

def ensure_topic(name, partitions=3, rf=1):
    futures = admin.create_topics([NewTopic(name, num_partitions=partitions, replication_factor=rf)])
    for n, f in futures.items():
        try: f.result()
        except Exception as e:
            if "already exists" not in str(e) and "TopicExistsError" not in str(type(e)):
                raise

ensure_topic(TOPIC)

# Produce a fresh batch of records so each demo below has something to read.
p = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all", "enable.idempotence": True})
for i in range(12):
    p.produce(TOPIC, key=f"CUST{i%4:04d}", value=f"event-{i:03d}")
p.flush()
print("produced 12 records")

## A minimum consumer

Four config keys are enough to start:

- **`bootstrap.servers`** — where to find the cluster.
- **`group.id`** — the name of the consumer group this consumer belongs to. The single most important config; the source of identity for rebalancing and offset storage.
- **`auto.offset.reset`** — what to do when this group has no committed offset (typically because it's brand new).
- **`enable.auto.commit`** — whether the client automatically commits offsets in the background (default `True`).

Below: subscribe, poll a few times, print what arrives, close.

In [ ]:
c = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": "demo-group-A",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,    # we'll explore commit modes manually below
})
c.subscribe([TOPIC])

print(f"{'partition':>9}  {'offset':>6}  {'key':<10}  value")
print("-" * 50)
read = 0
while read < 12:
    msg = c.poll(2.0)
    if msg is None: break
    if msg.error(): print("err:", msg.error()); continue
    print(f"{msg.partition():>9}  {msg.offset():>6}  {msg.key().decode():<10}  {msg.value().decode()}")
    read += 1

c.close()

## Consumer groups — parallelism without coordination

A consumer group is a set of consumers that share the work of reading a topic. The group coordinator (a broker, picked per group) assigns each partition to exactly one consumer in the group.

```text
  topic: payments (6 partitions)

  group: billing                                group: analytics
  ┌────────────┐                                ┌────────────┐
  │ consumer A │ → owns p0, p1                  │ consumer X │ → owns p0..p5
  │ consumer B │ → owns p2, p3                  └────────────┘
  │ consumer C │ → owns p4, p5
  └────────────┘
      3 consumers split 6 partitions               1 consumer owns all 6
```

Two things to internalize:

- **Each partition is read by exactly one consumer within a group.** That's how Kafka gives you parallelism without locks or coordination — each consumer owns a disjoint slice of the topic.
- **Different groups are independent.** Group `billing` and group `analytics` each get their own view of the topic, each with their own committed offsets. Adding a new group doesn't slow anything down or steal records from existing groups.

**The hard ceiling on parallelism is the partition count.** Six partitions, six consumers — perfect. Six partitions, eight consumers — two consumers sit idle. The only way past that ceiling is to add partitions to the topic (which, as you saw in notebook 01, breaks per-key ordering for any key that moves).

## The group coordinator and the heartbeat

Each group has a designated **group coordinator** — one broker in the cluster, picked deterministically from the `group.id` hash. The coordinator is responsible for:

- Tracking which consumers are alive in the group (via heartbeats).
- Running the rebalance protocol when membership changes.
- Persisting the group's committed offsets to the internal `__consumer_offsets` topic.

Three timeouts govern liveness, and you should know them by name:

| Setting | Default | Meaning |
|---|---|---|
| `session.timeout.ms` | `45000` | If the coordinator hasn't seen a heartbeat for this long, the consumer is declared dead and dropped from the group |
| `heartbeat.interval.ms` | `3000` | How often the consumer sends a heartbeat (separate background thread) |
| `max.poll.interval.ms` | `300000` | If your code doesn't call `poll()` for this long, the consumer is declared dead — even if heartbeats are still flowing |

Heartbeats run on a background thread, so a hung processing loop won't stop them. That's why `max.poll.interval.ms` exists — it's the safety net for slow processing. If your handler can take longer than five minutes on a record, raise this; if it takes more than that *because something is wrong*, you want the rebalance to kick in. The defaults assume "normal" record processing finishes well under 5 minutes.

## The rebalance protocol

Anything that changes group membership triggers a **rebalance**: a consumer joins, a consumer leaves, a consumer dies, the topic's partition count changes. The coordinator runs a small protocol that redistributes partitions across the surviving members.

Two flavors of rebalance exist, and the gap between them matters for production:

**Eager rebalance** (legacy, with the `range` and `round-robin` strategies):

1. *Every* consumer stops reading and revokes *all* its partitions.
2. The coordinator computes the new assignment.
3. Each consumer gets its new partitions and resumes.

During step 1–3, no partition is being read. This is the infamous **stop-the-world rebalance**: a multi-minute consumer outage when a fat group rebalances during a bad deploy.

**Cooperative (incremental) rebalance** (modern, with `cooperative-sticky`):

1. The coordinator computes the new assignment.
2. Only the *delta* is revoked — partitions that need to move to a different consumer.
3. Consumers keep reading partitions they still own throughout.

The stop-the-world window shrinks to the few partitions that actually move. This is what you want for any group of more than a handful of consumers.

## Assignment strategies — pick one

`partition.assignment.strategy` chooses how the coordinator hands partitions out.

| Strategy | Rebalance flavor | Behavior |
|---|---|---|
| `range` (legacy default) | Eager | Per topic, contiguous range of partitions per consumer. Imbalanced when partition count doesn't divide evenly |
| `roundrobin` | Eager | Spreads partitions evenly across consumers across all subscribed topics |
| `sticky` | Eager | Like `roundrobin`, but tries to minimize partition movement across rebalances |
| `cooperative-sticky` | **Cooperative** | Same balance properties as `sticky`, but uses the incremental rebalance protocol |

**The honest recommendation: use `cooperative-sticky` unless you have a reason not to.** It minimizes both partition movement *and* the stop-the-world window. The legacy `range` default exists for backward compatibility; new groups should opt in to cooperative.

Set it:

```python
Consumer({..., "partition.assignment.strategy": "cooperative-sticky"})
```

All consumers in the group must agree on the strategy — the coordinator picks the one shared by every member. A rolling migration from `range` to `cooperative-sticky` is allowed via a temporary `["cooperative-sticky", "range"]` list during the transition.

## Offsets revisited

Two distinct numbers live on the consumer side:

- **Current position** — the next offset `poll()` will return for a partition. Lives in the consumer process; advances every time a record is delivered.
- **Committed offset** — the offset *persisted to the broker* as "this group has processed everything before this point." When a consumer joins a group and gets assigned a partition, the broker hands back the committed offset, and the consumer resumes from there.

Committed offsets are stored in an internal Kafka topic, `__consumer_offsets`, with one record per `(group, topic, partition)`. It's just another compacted topic — you can read it like any other, although in practice you use `kafka-consumer-groups.sh` to inspect group state.

Think of it this way: the consumer's *position* is a runtime cursor; the *committed offset* is a save game. A consumer crash forgets the cursor but preserves the save.

## `auto.offset.reset` — what "new" means

When a consumer joins a group for the first time, or its committed offset has been deleted by retention, the consumer has nowhere to start from. `auto.offset.reset` picks the policy:

- **`earliest`** — start at offset 0 (or whatever's earliest still in the log after retention). Used when you want to process everything you can see.
- **`latest`** — start at the end of the log. Used when you only care about new records from now on. **This is the default.**
- **`none`** — throw `NoOffsetForPartitionError`. Used when missing offsets indicate a real bug worth surfacing rather than silently picking a side.

**The trap to remember:** `auto.offset.reset` only matters when there's *no committed offset*. Once a group commits even one offset, the setting is irrelevant — the group resumes from its commit. Beginners often think "changing it to earliest will reprocess everything," then are surprised when nothing happens. To actually reprocess, you have to seek (see the section below) or use a fresh `group.id`.

## Commit modes — auto vs manual, sync vs async

Four combinations. Pick deliberately.

**Auto commit (`enable.auto.commit=True`, default).** The client periodically commits offsets in the background, every `auto.commit.interval.ms` (default 5 seconds). The committed offset is *the position returned by the most recent `poll()`* — so the commit advances ahead of your processing. If your code crashes after `poll()` but before finishing the record, the next consumer starts past the unprocessed record. **At-most-once semantics by accident.**

**Manual commit, synchronous (`commit()` with no args).** You call `commit()` after processing a batch. Blocks until the broker acknowledges. Slow but correct: the commit guarantees the broker has the offset persisted.

**Manual commit, asynchronous (`commit(asynchronous=True)`).** You call `commit(asynchronous=True)` after processing. Returns immediately; the commit happens in the background. Fast, but if the consumer crashes between the call and the actual commit, the offset is lost.

**Common production pattern:** async commit during the steady-state loop (cheap), then one sync commit on shutdown (correct on the way out). Below: a manual-commit loop.

*Note: `confluent-kafka` uses the keyword argument `asynchronous=` (not `async=`, which is a reserved word in Python).*

In [ ]:
c = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": "demo-group-manual-commit",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
    "partition.assignment.strategy": "cooperative-sticky",
})
c.subscribe([TOPIC])

buffered = []  # process in small batches, commit once per batch
BATCH = 5
processed = 0

while processed < 12:
    msg = c.poll(2.0)
    if msg is None: break
    if msg.error(): continue
    buffered.append(msg)
    if len(buffered) >= BATCH:
        # "process" the batch (here, just print)
        for m in buffered:
            print(f"processed p={m.partition()} o={m.offset()} v={m.value().decode()}")
        # commit after processing — at-least-once semantics
        c.commit(asynchronous=False)
        processed += len(buffered)
        buffered.clear()

# Drain anything still in the buffer.
for m in buffered:
    print(f"processed p={m.partition()} o={m.offset()} v={m.value().decode()}")
if buffered:
    c.commit(asynchronous=False)
c.close()

## Delivery semantics on the consumer side

The producer side has `acks` and idempotence; the consumer side has commit ordering. The choice between **at-most-once** and **at-least-once** comes down to *when* you commit relative to *when* you process:

- **Commit before processing → at-most-once.** A crash after commit means the next consumer skips a record. Records can be lost, never duplicated.
- **Commit after processing → at-least-once.** A crash after processing but before commit means the next consumer reprocesses. Records can be duplicated, never lost. **This is the default and almost always the right choice.**

Because duplicates are inevitable in at-least-once mode, **your downstream must be idempotent** — reprocessing the same record twice should yield the same result. Common patterns: deduplication keys, upsert/merge writes, idempotent HTTP endpoints.

**Exactly-once** is the third option, and it's not a single setting. It requires:

1. **Producer side:** `transactional.id` (notebook 02) so writes are transactional.
2. **Consumer side:** `isolation.level=read_committed` so aborted records are skipped.
3. **A consume-process-produce pattern** where the *offset commit and the produced output are in the same transaction*. The producer API method `send_offsets_to_transaction` does that — the offsets are committed only if the transaction commits.

For a plain consume-process loop (no downstream produce), exactly-once collapses to "at-least-once plus idempotent processing." That is by far the most common shape in production.

## `isolation.level=read_committed`

When the producer side uses transactions, the broker writes the transaction's records to the log *immediately* — and then marks the transaction committed or aborted later. A naive consumer would see records from an aborted transaction.

`isolation.level` picks how the consumer handles that:

- **`read_uncommitted`** (default) — return every record on disk, including aborted-transaction records. Fastest, no protocol overhead.
- **`read_committed`** — skip records from aborted transactions, and don't return records from in-flight (not-yet-committed) transactions. Required for the exactly-once recipe.

Below: re-run the transaction demo from notebook 02, then read both topics with `read_committed`. The aborted `ORD-002` records are filtered out.

Note: `read_committed` introduces a small latency tax — the consumer can't return records past an in-flight transaction's first record until it knows the outcome. Most workloads don't notice; latency-sensitive ones should measure.

In [ ]:
ensure_topic("tx-orders", partitions=1)
ensure_topic("tx-payments", partitions=1)

txp = Producer({"bootstrap.servers": BOOTSTRAP, "transactional.id": "nb03-tx-demo"})
txp.init_transactions()

txp.begin_transaction()
txp.produce("tx-orders",   key="ORD-A", value="created")
txp.produce("tx-payments", key="ORD-A", value="authorized")
txp.commit_transaction()

txp.begin_transaction()
txp.produce("tx-orders",   key="ORD-B", value="created")
txp.produce("tx-payments", key="ORD-B", value="FAILED")
txp.abort_transaction()

rc = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": "nb03-rc-reader",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
    "isolation.level": "read_committed",
})
rc.subscribe(["tx-orders", "tx-payments"])

print("records visible to read_committed:")
seen = 0
while seen < 2:
    msg = rc.poll(2.0)
    if msg is None: break
    if msg.error(): continue
    print(f"  topic={msg.topic():<12} key={msg.key().decode()} value={msg.value().decode()}")
    seen += 1
rc.close()
print("ORD-B from the aborted transaction is filtered out")

## Static membership — surviving restarts without rebalances

By default, every consumer process is anonymous from the coordinator's perspective — when a process restarts, it looks like a new member, and a rebalance runs. For a 100-pod consumer deployment, a rolling restart can trigger 100 rebalances.

**Static membership** (`group.instance.id`) gives each consumer a stable identity. When a consumer with a known `group.instance.id` disconnects and reconnects within `session.timeout.ms`, the coordinator treats it as the same member and *does not rebalance*. The partitions are held for that instance ID until either the consumer comes back or the session times out.

Two practical notes:

- Set `session.timeout.ms` high enough to cover a normal restart — usually one to two minutes.
- The instance ID must be **unique per process**. In Kubernetes, this is naturally the pod name; in a static deployment, it's the host name. Re-using an instance ID across two live processes fences the older one.

```python
Consumer({
    ...
    "group.id": "billing",
    "group.instance.id": os.environ["HOSTNAME"],
    "session.timeout.ms": 60000,
})
```

Combined with `cooperative-sticky`, static membership is the modern recipe for big consumer groups in production: rolling restarts cause near-zero rebalance churn.

## `subscribe()` vs `assign()`

Two ways to tell a consumer what to read:

- **`subscribe(["topic-a", "topic-b"])`** — join a group; the coordinator hands you partitions and rebalances when membership changes. The default and what you want 95% of the time.
- **`assign([TopicPartition("topic-a", 0), TopicPartition("topic-a", 1)])`** — pin to specific partitions manually. No group membership, no rebalance, no offset auto-commit interaction with a group. You take responsibility for fault tolerance — if your consumer dies, nothing reassigns its partitions.

When to reach for `assign()`:

- One-off scripts that need to read a specific partition.
- Stateful stream-processing frameworks (Kafka Streams, Flink) that do their own partition assignment.
- Tests that want exact, deterministic positioning.

## `seek()` — replay, skip, time-travel

Once partitions are assigned (either via `subscribe()` after the first `poll()`, or directly via `assign()`), `seek()` moves the consumer's position to an arbitrary offset.

Three common patterns:

- **Seek to beginning** — `seek(TopicPartition(topic, partition, OFFSET_BEGINNING))`. Replay everything in the partition.
- **Seek to end** — `seek(TopicPartition(topic, partition, OFFSET_END))`. Skip past everything currently in the log; only read what arrives next.
- **Seek by timestamp** — `offsets_for_times([(tp, timestamp_ms)])` returns the first offset at or after each timestamp, which you then `seek()` to. Useful for "start from yesterday at midnight" replays.

Below: seek a fresh consumer to the beginning of partition 0 and read just that partition.

In [ ]:
c = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": "demo-seek-reader",
    "enable.auto.commit": False,
})
# assign directly so we can seek immediately (no rebalance to wait for)
tp = TopicPartition(TOPIC, 0, OFFSET_BEGINNING)
c.assign([tp])

print("reading partition 0 from the beginning:")
for _ in range(5):
    msg = c.poll(2.0)
    if msg is None: break
    if msg.error(): continue
    print(f"  o={msg.offset()} k={msg.key().decode()} v={msg.value().decode()}")
c.close()

## `pause()` / `resume()` — consumer-side back-pressure

Sometimes a downstream system (database, HTTP API) is overloaded and you need the consumer to stop pulling new records — but you can't just stop calling `poll()`, because that would trigger a rebalance.

`pause([TopicPartition, ...])` tells the consumer "don't fetch from these partitions on the next polls." `resume([...])` undoes it. The consumer keeps heartbeating and stays in the group; it just stops fetching.

Use cases:

- Database connection pool exhausted → pause until pool recovers.
- Buffer to a downstream queue full → pause until queue drains.
- Per-partition back-pressure — pause only the partitions that point to the slow downstream tenant.

The mental model: `pause()` is to a consumer what an inbound rate-limit is to an HTTP client. It's how you avoid blowing yourself up when downstream slows down.

## Common gotchas

- **Blocking inside the poll loop.** Anything slow inside the loop pushes you toward `max.poll.interval.ms`. Offload heavy work or raise the limit deliberately — don't drift into rebalance storms.
- **Auto-commit + side effects.** With `enable.auto.commit=True`, a crash mid-processing skips records. If your handler writes to a database, switch to manual commit after the write succeeds.
- **Forgetting to `close()`.** A clean `close()` commits offsets (if auto-commit is on) and sends a `LeaveGroup` request so the coordinator rebalances immediately. Skipping it makes the group wait `session.timeout.ms` before noticing.
- **Reusing a `group.id` for unrelated workloads.** Two unrelated consumer applications with the same `group.id` will split the topic between them — each sees only half the records. Group IDs are *application identity*, not labels.
- **Expecting `auto.offset.reset=earliest` to reprocess.** It only kicks in when there's no committed offset. Use `seek()` or a new `group.id` to actually replay.
- **Too many partitions per consumer.** A single consumer assigned hundreds of partitions becomes the bottleneck. Either add consumers or split the topic.
- **`group.instance.id` collisions.** Two processes with the same static ID fence each other — older one starts seeing fenced errors. Use the pod/host name.

## What's next

You can now produce and consume safely, with a clear picture of rebalances, commits, and the exactly-once recipe end to end.

- **Notebook 04 — Topics, Partitions & Storage.** How to size a partition count, retention policies (time vs size), log compaction, segment files on disk, tiered storage.
- **Notebook 05 — Schema Registry & Serialization.** Why opaque bytes is a footgun at scale, Avro / Protobuf / JSON Schema, schema evolution, the wire format.
- **Notebook 06 — Kafka Connect.** Pre-built producers and consumers running as long-lived workers — source connectors, sink connectors, SMTs.

Producers and consumers are the API. Topic design and serialization are how you make that API scale.